# From Words to Meanings — Try it in PyTorch

This is an **optional** hands-on companion to [Chapter 5: From Words to Meanings](https://learnai.robennals.org/embeddings). I'll explore a real 300-dimensional GloVe embedding, project words onto a spectrum line, do vector arithmetic on word meanings, see how a real tokenizer splits text, and then train my own word embeddings from scratch using word2vec.

*New to PyTorch? See the [PyTorch appendix](https://learnai.robennals.org/appendix-pytorch) for a beginner-friendly introduction.*

In [ ]:
import torch
import torch.nn as nn
import urllib.request
import os
import zipfile
import matplotlib.pyplot as plt

## Exploring a Real Embedding

I'll start with a real pre-trained embedding: **GloVe**, created by Stanford. Each word is a point in 300-dimensional space, trained on billions of words of text. The full embedding contains 400,000 words.

First I download and load it. The full zip is ~862MB; if running in Colab the download takes a minute or two.

In [ ]:
glove_path = 'glove.6B.300d.txt'
if not os.path.exists(glove_path):
    print('Downloading GloVe embeddings (~862MB zip — this takes a minute)...')
    urllib.request.urlretrieve(
        'https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip',
        'glove.6B.zip',
    )
    with zipfile.ZipFile('glove.6B.zip', 'r') as z:
        z.extract('glove.6B.300d.txt')
    print('Done!')

print('Loading 300d embeddings into a dict (this takes a couple of minutes)...')
word_vecs = {}
with open(glove_path, 'r') as f:
    for line in f:
        parts = line.split()
        word = parts[0]
        vec = torch.tensor([float(x) for x in parts[1:]])
        word_vecs[word] = vec
print(f'Loaded {len(word_vecs)} word vectors of dim {len(next(iter(word_vecs.values())))}')

In [ ]:
def nearest(word, top_k=5):
    """Find top-k nearest neighbors of `word` by cosine similarity."""
    if word not in word_vecs:
        return []
    target = word_vecs[word]
    scored = []
    for w, v in word_vecs.items():
        if w == word:
            continue
        sim = torch.cosine_similarity(target.unsqueeze(0), v.unsqueeze(0)).item()
        scored.append((w, sim))
    scored.sort(key=lambda x: -x[1])
    return scored[:top_k]

for seed in ['dog', 'guitar', 'queen']:
    print(f'\n--- nearest to {seed} ---')
    for w, s in nearest(seed):
        print(f'  {w:15s} {s:.3f}')

## Directions Have Meaning

The embedding doesn't just group similar words together — *directions* in the space carry meaning too. The chapter's `WordPairSpectrumWidget` finds words that genuinely lie on the line between two endpoints (not just words similar to one endpoint).

I'll implement the same algorithm: for each candidate word, project it onto the line between two endpoint words. Keep words whose projection lies *between* the endpoints (parameter `t` in [0.05, 0.95]) and that don't sit too far off the line.

In [ ]:
def spectrum(word_a, word_b, candidates, top_k=10, t_min=0.05, t_max=0.95):
    """Find words whose embedding lies along the line from word_a to word_b."""
    vec_a = word_vecs[word_a]
    vec_b = word_vecs[word_b]
    line_dir = vec_b - vec_a
    line_len_sq = (line_dir ** 2).sum()

    results = []
    for w in candidates:
        if w in (word_a, word_b) or w not in word_vecs:
            continue
        v = word_vecs[w]
        offset = v - vec_a
        t = ((offset * line_dir).sum() / line_len_sq).item()
        if not (t_min < t < t_max):
            continue
        proj = vec_a + t * line_dir
        perp = float(((v - proj) ** 2).sum().sqrt())
        results.append((w, t, perp))

    # Sort by perpendicular distance (closest to the line first), then take top_k
    # and re-sort by t so the spectrum reads in order.
    results.sort(key=lambda r: r[2])
    selected = results[:top_k]
    selected.sort(key=lambda r: r[1])
    return selected

# Use a candidate pool of 50k most common GloVe words (rare words mostly add noise)
candidate_pool = list(word_vecs.keys())[:50_000]

for a, b in [('rabbit', 'elephant'), ('salad', 'cake'), ('tiny', 'huge')]:
    print(f'\n--- spectrum from {a} to {b} ---')
    for w, t, perp in spectrum(a, b, candidate_pool):
        print(f'  t={t:.2f}  perp={perp:.2f}  {w}')

## Adding Meaning

If directions in embedding space carry meaning, then adding a vector to a word moves its meaning along that direction. I can extract a transformation as a subtraction (`elephant − mouse` = "bigger animal") and apply it as an addition.

Below are the four patterns the chapter highlights: bigger-animal, capital-of, gender-swap, and present-participle. Each is computed as `b − a + c`.

In [ ]:
def analogy(a, b, c, top_k=3):
    """a is to b as c is to ???  →  return top-k candidates by cosine similarity."""
    target = word_vecs[b] - word_vecs[a] + word_vecs[c]
    scored = []
    for w, v in word_vecs.items():
        if w in (a, b, c):
            continue
        sim = torch.cosine_similarity(target.unsqueeze(0), v.unsqueeze(0)).item()
        scored.append((w, sim))
    scored.sort(key=lambda x: -x[1])
    return scored[:top_k]

patterns = [
    ('bigger animal',     'mouse',  'elephant', 'rabbit'),
    ('capital of',        'france', 'paris',    'japan'),
    ('gender swap',       'man',    'king',     'woman'),
    ('present participle','walk',   'walking',  'run'),
]

for label, a, b, c in patterns:
    top = analogy(a, b, c)
    print(f'{label}: {b} - {a} + {c} =')
    for w, s in top:
        print(f'  {w:15s} {s:.3f}')
    print()

## From Words to Tokens

Real language models don't treat each whole word as one unit — they break text into **tokens** that can be whole words, parts of words, or punctuation. I'll use **tiktoken**, OpenAI's tokenizer for GPT-4. Notice how common words become a single token while rare or made-up words get split into recognizable parts.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding('cl100k_base')  # the GPT-4 tokenizer

def show_tokens(text):
    ids = enc.encode(text)
    tokens = [enc.decode([i]) for i in ids]
    print(f'  text:   {text!r}')
    print(f'  tokens: {tokens}')
    print(f'  ids:    {ids}')
    print()

for text in [
    'superman, superb, superlative',
    'qwertyflorp',
    'Hello, world! How are you?',
    'The cat sat on the mat.',
]:
    show_tokens(text)

## Where Do Embeddings Come From?

GloVe is **pre-trained**. Where do those vectors actually come from? I'll train my own embeddings from scratch using **word2vec** (skip-gram with negative sampling) — the same core idea behind modern embedding layers, just simpler. Word2vec learns by trying to predict which words appear near which other words.

I'll train on **text8** — Matt Mahoney's standard ~100MB Wikipedia text dump, the canonical tiny corpus for word2vec demos. The download is ~30MB compressed. Training takes a few minutes (faster on a GPU; switch the Colab runtime to **T4 GPU** for ~5x speedup).

In [ ]:
text8_path = 'text8'
if not os.path.exists(text8_path):
    print('Downloading text8.zip (~30MB)...')
    urllib.request.urlretrieve('https://mattmahoney.net/dc/text8.zip', 'text8.zip')
    with zipfile.ZipFile('text8.zip', 'r') as z:
        z.extract('text8')
    print('Done!')

print('Reading text8...')
with open(text8_path, 'r') as f:
    text = f.read()
tokens = text.split()
print(f'  {len(tokens):,} total tokens')
print(f'  first 20 tokens: {tokens[:20]}')

In [ ]:
from collections import Counter

VOCAB_SIZE = 10_000
EXAMPLE_WORDS = ['the', 'dog', 'cat', 'fish', 'car', 'apple', 'king', 'piano']

counts = Counter(tokens)
most_common = [w for w, _ in counts.most_common(VOCAB_SIZE)]

# Make sure all chapter example words are in the vocab even if they're rare
for w in EXAMPLE_WORDS:
    if w not in most_common:
        most_common.append(w)
        print(f'  added {w!r} (rank {[w_ for w_,_ in counts.most_common()].index(w)})')

word_to_id = {w: i for i, w in enumerate(most_common)}
id_to_word = most_common
vocab_size = len(word_to_id)
print(f'\nVocab size: {vocab_size}')

# Convert tokens to ids, dropping OOV
token_ids = [word_to_id[w] for w in tokens if w in word_to_id]
print(f'In-vocab tokens: {len(token_ids):,} ({len(token_ids)/len(tokens):.1%} of corpus)')

In [ ]:
EMBED_DIM = 100
WINDOW = 5
NEG_SAMPLES = 5

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Training on {device}')
if device == 'cpu':
    print('  (For ~5x speedup, switch the Colab runtime to T4 GPU.)')

class SkipGramNeg(nn.Module):
    """Skip-gram with negative sampling: each word has two embeddings,
    one as a center word and one as a context word."""

    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.in_emb = nn.Embedding(vocab_size, embed_dim)
        self.out_emb = nn.Embedding(vocab_size, embed_dim)
        # Initialize input embeddings small, output to zero (Mikolov-style)
        nn.init.uniform_(self.in_emb.weight, -0.5/embed_dim, 0.5/embed_dim)
        nn.init.zeros_(self.out_emb.weight)

    def forward(self, centers, contexts, negatives):
        # centers: (B,) ; contexts: (B,) ; negatives: (B, K)
        c = self.in_emb(centers)            # (B, D)
        ctx = self.out_emb(contexts)        # (B, D)
        neg = self.out_emb(negatives)       # (B, K, D)
        pos_score = (c * ctx).sum(dim=1)                      # (B,)
        neg_score = torch.bmm(neg, c.unsqueeze(2)).squeeze(2) # (B, K)
        # log_sigmoid(x) = -softplus(-x); use softplus form for MPS compatibility
        loss = torch.nn.functional.softplus(-pos_score).mean() \
               + torch.nn.functional.softplus(neg_score).mean()
        return loss

model = SkipGramNeg(vocab_size, EMBED_DIM).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')

In [ ]:
import random
import time

EPOCHS = 3
BATCH_SIZE = 1024
LR = 0.005

# Pre-build skip-gram pairs from a SUBSAMPLE of the corpus to keep training time bounded.
# Subsampling frequent words (Mikolov's trick) accelerates training and improves quality.
import math
freq = torch.tensor([counts.get(id_to_word[i], 0) for i in range(vocab_size)], dtype=torch.float32)
p_word = freq / freq.sum()
# Subsample: keep word with probability sqrt(t / f) where t is a small constant
t_subsample = 1e-4
keep_prob = torch.clamp((torch.sqrt(t_subsample / p_word) + t_subsample / p_word), max=1.0)

rng = random.Random(0)
filtered_ids = [i for i in token_ids if rng.random() < keep_prob[i].item()]
print(f'After subsampling: {len(filtered_ids):,} tokens (from {len(token_ids):,})')

# Negative sampling distribution: unigram^0.75
neg_probs = (freq ** 0.75)
neg_probs = neg_probs / neg_probs.sum()

# Build all (center, context) pairs once
print('Building skip-gram pairs...')
centers, contexts = [], []
for i, c_id in enumerate(filtered_ids):
    # Random window size in [1, WINDOW] (Mikolov-style — biases toward closer context)
    w = rng.randint(1, WINDOW)
    lo, hi = max(0, i - w), min(len(filtered_ids), i + w + 1)
    for j in range(lo, hi):
        if j == i: continue
        centers.append(c_id)
        contexts.append(filtered_ids[j])
centers = torch.tensor(centers, dtype=torch.long)
contexts = torch.tensor(contexts, dtype=torch.long)
n_pairs = len(centers)
print(f'Total skip-gram pairs: {n_pairs:,}')

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

model.train()
for epoch in range(EPOCHS):
    t0 = time.time()
    perm = torch.randperm(n_pairs)
    total_loss = 0.0
    n_batches = 0
    for i in range(0, n_pairs, BATCH_SIZE):
        batch = perm[i:i+BATCH_SIZE]
        c = centers[batch].to(device)
        x = contexts[batch].to(device)
        n = torch.multinomial(neg_probs, len(batch) * NEG_SAMPLES, replacement=True)
        n = n.view(len(batch), NEG_SAMPLES).to(device)
        loss = model(c, x, n)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    print(f'Epoch {epoch+1}/{EPOCHS}  loss={total_loss/n_batches:.4f}  time={time.time()-t0:.1f}s')

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

# Pull the input embeddings (the conventional 'word embedding' from skip-gram)
trained_vecs = model.in_emb.weight.detach().cpu().numpy()

# Pick which words to plot: chapter's 8 example tokens plus a few neighbors so clusters are visible
extra_words = ['puppy', 'kitten', 'truck', 'apple', 'banana', 'queen', 'guitar', 'piano', 'violin', 'horse']
plot_words = list(dict.fromkeys(EXAMPLE_WORDS + extra_words))  # dedupe, preserve order
plot_words = [w for w in plot_words if w in word_to_id]
ids = [word_to_id[w] for w in plot_words]
vecs_to_plot = trained_vecs[ids]

# PCA to 2D for visualization
pca = PCA(n_components=2)
coords = pca.fit_transform(vecs_to_plot)

plt.figure(figsize=(9, 7))
plt.scatter(coords[:, 0], coords[:, 1], s=40)
for w, (x, y) in zip(plot_words, coords):
    plt.annotate(w, (x, y), xytext=(5, 5), textcoords='offset points', fontsize=11)
plt.title('Word2vec embeddings trained on text8 (PCA to 2D)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Sanity check: cosine similarity in the FULL learned space (not the 2D projection)
def cossim(a, b):
    return float(torch.cosine_similarity(
        torch.tensor(trained_vecs[word_to_id[a]]).unsqueeze(0),
        torch.tensor(trained_vecs[word_to_id[b]]).unsqueeze(0),
    ))

print('\nQuick sanity check (cosine similarity in the trained 100d space):')
print(f'  cossim(dog, cat) = {cossim("dog", "cat"):.3f}')
print(f'  cossim(dog, car) = {cossim("dog", "car"):.3f}')
print(f'  cossim(cat, car) = {cossim("cat", "car"):.3f}')
print('Expected: dog/cat similarity > dog/car and cat/car similarity (animals cluster).')

---

*This notebook accompanies [Chapter 5: From Words to Meanings](https://learnai.robennals.org/embeddings). Next up: [Chapter 6: Understanding by Predicting](https://learnai.robennals.org/next-word-prediction) — where I'll build a next-word predictor that uses these embedding ideas to generalize across similar words.*

*New to PyTorch? See the [PyTorch appendix](https://learnai.robennals.org/appendix-pytorch) for a beginner-friendly introduction.*